# 02. Random survival forest: hyperparameter search and final model

* Hyperparameters were searched with Optuna (200 trials) by maximizing the validation C-index
  (search ranges: `n_estimators` 100-400, `min_samples_split` 5-30, `min_samples_leaf` 5-30,
  `max_features` sqrt/log2; Supplementary Table 2).
* `RUN_OPTUNA = False` (default) trains the final model directly with the selected hyperparameters
  (100 trees, min_samples_split 10, min_samples_leaf 5, max_features log2, random_state 42).
  Set `RUN_OPTUNA = True` to repeat the search.
* The fitted model is saved to `outputs/RSF_model.joblib`.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sksurv.util import Surv

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH    = Path("../data/dummy_data.csv")     # replace with your own data
OUTPUT_DIR   = Path("../outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
PREPROC_PATH = OUTPUT_DIR / "preprocessor.joblib"  # written by 01_data_preprocessor
MODEL_PATH   = OUTPUT_DIR / "RSF_model.joblib"     # written by 02_RSF_train
RANDOM_SEED  = 42

TIME_COL, EVENT_COL, ID_COLS = "OS", "event", ["SubjectKey"]   # OS in days

def load_splits(path=DATA_PATH):
    """Return {"Train"|"Val"|"Test": (X_raw, y)} using the column split_3."""
    df = pd.read_csv(path)
    df[EVENT_COL] = df[EVENT_COL].astype(bool)
    out = {}
    for name, key in [("Train", "train"), ("Val", "val"), ("Test", "test")]:
        d = df[df["split_3"] == key]
        out[name] = (d.drop(columns=ID_COLS + [TIME_COL, EVENT_COL]),
                     Surv.from_arrays(d[EVENT_COL], d[TIME_COL]))
    return out

In [ ]:
class Pred:
    """Predictions of a fitted RSF for one evaluated set."""
    def __init__(self, model, X, y):
        self.risk  = model.predict(X)
        self.S     = model.predict_survival_function(X, return_array=True)
        self.T     = model.unique_times_
        self.event = y["event"]
        self.time  = y["time"]

    def surv_at(self, rows, grid):
        """Step-function survival S(t) at the given times for the given rows."""
        grid = np.atleast_1d(np.asarray(grid, dtype=float))
        idx  = np.searchsorted(self.T, grid, side="right") - 1
        out  = np.ones((len(rows), len(grid)))
        ok   = idx >= 0
        out[:, ok] = self.S[rows][:, idx[ok]]
        return out


from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc, integrated_brier_score

TIME_POINTS = [730, 1825, 2555, 3650]                 # days
TP_LABELS   = ["2-year", "5-year", "7-year", "10-year"]

def metrics(pred, y_ref):
    """C-index (Harrell), IBS and 2/5/7/10-year TD-AUC; censoring distribution from y_ref (training set).
    IBS: 100-point grid from the minimum to the maximum follow-up of the evaluated set (end point excluded).
    TD-AUC: cumulative/dynamic AUC with IPCW (Uno); score = 1 - S(t)."""
    ev, tm = pred.event, pred.time
    y = Surv.from_arrays(ev, tm)
    rows = np.arange(len(tm))
    out = {"C-index": concordance_index_censored(ev, tm, pred.risk)[0]}
    grid = np.linspace(tm.min(), tm.max(), 100, endpoint=False)
    out["IBS"] = integrated_brier_score(y_ref, y, pred.surv_at(rows, grid), grid)
    p_death = 1.0 - pred.surv_at(rows, TIME_POINTS)
    for k, lbl in enumerate(TP_LABELS):
        out[f"{lbl} AUC"] = cumulative_dynamic_auc(y_ref, y, p_death[:, k], [TIME_POINTS[k]])[0][0]
    return out

In [ ]:
from sksurv.ensemble import RandomSurvivalForest

RUN_OPTUNA = False
N_TRIALS   = 200
RSF_PARAMS = {"n_estimators": 100, "min_samples_split": 10,      # Supplementary Table 2
              "min_samples_leaf": 5, "max_features": "log2"}

splits = load_splits()
preprocessor = joblib.load(PREPROC_PATH)   # from 01_data_preprocessor.ipynb
X = {s: preprocessor.transform(splits[s][0]) for s in splits}
Y = {s: splits[s][1] for s in splits}

def train_rsf(params):
    return RandomSurvivalForest(**params, n_jobs=-1, random_state=RANDOM_SEED).fit(X["Train"], Y["Train"])

In [ ]:
if RUN_OPTUNA:
    import optuna

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 400, step=50),
            "min_samples_split": trial.suggest_int("min_samples_split", 5, 30),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 5, 30),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
        m = train_rsf(params)
        return concordance_index_censored(Y["Val"]["event"], Y["Val"]["time"], m.predict(X["Val"]))[0]

    study = optuna.create_study(direction="maximize", study_name="RSF_Opt")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f"Best validation C-index: {study.best_value:.4f}")
    params = study.best_params
else:
    params = RSF_PARAMS
print("Hyperparameters:", params)

In [ ]:
model = train_rsf(params)
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH)

# C-index and IBS by split (Table 2); censoring distribution from the training set
rows = {}
for s in ("Train", "Val", "Test"):
    m = metrics(Pred(model, X[s], Y[s]), Y["Train"])
    rows[s] = {"C-index": round(m["C-index"], 4), "IBS": round(m["IBS"], 4)}
print(pd.DataFrame(rows).to_string())